# 07c — S5P wind-aligned plume stack (hardened)
Reads scene catalog if present; otherwise writes skipped outputs.

In [ ]:
import os, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import yaml

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path):
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

def safe_json_response(resp):
    try:
        return resp.json(), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

def env(name):
    v = os.getenv(name)
    return v.strip() if isinstance(v, str) and v.strip() else None

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(10):
        if (cur / "configs").exists() and (cur / "notebooks").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

ROOT = find_repo_root(Path.cwd())
CONFIGS = ROOT / "configs"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
cfg = yaml.safe_load((CONFIGS / "run.yml").read_text(encoding="utf-8")) if (CONFIGS / "run.yml").exists() else {}
sites = cfg.get("sites", [])
run_cfg = cfg.get("run", {})
DATE_FROM = run_cfg.get("date_from", "2025-01-01")
DATE_TO = run_cfg.get("date_to", "2025-01-31")

def manifest_base(step):
    return {
        "step": step,
        "created_at_utc": utc_now(),
        "repo_root": str(ROOT),
        "configs": [{"path": str(CONFIGS / "run.yml"), "sha256": sha256_file(CONFIGS / "run.yml")}] if (CONFIGS / "run.yml").exists() else [],
        "artifacts": [],
        "notes": [],
        "status": "ok",
    }

def add_artifact(mf, p: Path, meta=None):
    if p.exists():
        row = {"path": str(p), "sha256": sha256_file(p)}
        if meta:
            row["meta"] = meta
        mf["artifacts"].append(row)

In [ ]:
step = "07c_s5p_wind_aligned_plume_stack_newhaven"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

mf = manifest_base(step)
cat = OUTPUTS / "07b_s5p_cdse_window_extract" / "scene_catalog.csv"
if not cat.exists():
    df = pd.DataFrame(columns=["site_id","site_name","plume_stack_score","status"])
    df.to_csv(out / "plume_stack_summary.csv", index=False)
    add_artifact(mf, out / "plume_stack_summary.csv", {"rows": 0})
    mf["status"] = "skipped_missing_upstream"
    mf["notes"].append("07b scene catalog missing; plume stack skipped.")
else:
    src = pd.read_csv(cat)
    if src.empty or ("scene_count" in src.columns and src["scene_count"].fillna(0).sum() == 0):
        df = src[["site_id","site_name"]].copy() if set(["site_id","site_name"]).issubset(src.columns) else pd.DataFrame(columns=["site_id","site_name"])
        if df.empty:
            df = pd.DataFrame(columns=["site_id","site_name"])
        df["plume_stack_score"] = 0.0
        df["status"] = "no_scenes_available"
    else:
        df = src[["site_id","site_name"]].copy()
        df["plume_stack_score"] = 0.0
        df["status"] = "placeholder"
        mf["notes"].append("Hardened notebook avoids fatal crash; plume stacking logic should be restored after CDSE extraction is stable.")
    df.to_csv(out / "plume_stack_summary.csv", index=False)
    add_artifact(mf, out / "plume_stack_summary.csv", {"rows": int(len(df))})
write_json(out / "manifest.json", mf)
display(pd.read_csv(out / "plume_stack_summary.csv"))